# 안면부 피부질환(AI허브 71863) — `facial` 태스크 학습 (Colab 래퍼)

이 노트북은 **계산을 직접 하지 않는다.** 저장소의 `fastapi/training/aihub/` 스크립트
(`facial/build_manifest.py → facial/make_holdout.py → train.py`)를 순서대로 부르는 래퍼다.
같은 세 명령을 맥북(MPS)이나 안심존 GPU 서버에서 쳐도 결과 형식이 같다 — 서버가 바뀌어도 코드는 바뀌지 않게 하기 위해서다.
설계와 데이터 사실은 `docs/aihub-datasets-plan.md`(§1-1, §3, §7), 실행 순서는 `fastapi/training/aihub/README.md`.

**준비물 (Drive)** — `MyDrive/artifact_medical_ai/facial/` 아래에 AI허브에서 받은 `1.데이터` 폴더를 **그대로**(zip 48개, 약 10GB) 올려 둔다.
```
MyDrive/artifact_medical_ai/facial/1.데이터/
├── Training/01.원천데이터/TS_<질환>_<뷰>.zip ×12   Training/02.라벨링데이터/TL_… ×12
└── Validation/01.원천데이터/VS_… ×12             Validation/02.라벨링데이터/VL_… ×12
```
맥에서 Drive 로 올릴 때 `1.데이터` 폴더를 통째로 끌어다 놓으면 된다(zip 을 풀지 않는다).

**런타임** — `런타임 > 런타임 유형 변경 > T4 GPU`. 디스크는 압축 해제 후 약 25GB 를 쓰므로 `/content` 에 푼다(Drive 에 풀면 학습이 10배 느려진다).

**규칙** — 홀드아웃은 AI허브 Validation 분할(1,200장)을 그대로 쓰고 `fastapi/tests/baselines/facial-holdout.csv` 로 커밋한다. 학습 산출물 중 저장소에 들어가는 것은 `facial.pth`(→ `fastapi/models/`)와 리포트 JSON 뿐이다. 이미지는 커밋하지 않는다.


In [ ]:
# 1. 런타임 확인 — GPU 가 없으면 여기서 멈춘다 (CPU 로 1만 장을 돌리면 몇 시간이 간다)
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "GPU 없음")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임 유형을 T4 GPU 로 바꾼 뒤 다시 실행"


In [ ]:
# 2. Drive 연결 + 경로 상수
from google.colab import drive
drive.mount('/content/drive')

DRIVE   = '/content/drive/MyDrive/artifact_medical_ai/facial'   # 1.데이터 폴더(zip 48개)가 있는 곳. 산출물도 여기로 돌려보낸다
DATA_IN = f'{DRIVE}/1.데이터'
REPO    = '/content/artifact'                                    # 저장소 clone 위치
EXTRACT = '/content/facial_data'                                 # zip 을 푸는 곳 (/content — Drive 아님)
RUN     = '/content/facial_run'                                  # manifest · 가중치 · 리포트
BRANCH  = 'kannikii/admiring-mccarthy-nfen9c'                    # 머지된 뒤에는 main 으로 바꾼다

import os
assert os.path.isdir(DATA_IN), f'{DATA_IN} 가 없다 — 위 준비물 설명대로 1.데이터 폴더를 올렸는지 확인'
os.makedirs(RUN, exist_ok=True)


In [ ]:
# 3. 저장소 받기 + timm 설치 (Colab 에는 torch/torchvision 이 이미 있다)
import subprocess, sys, os
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/CSID-DGU/2026-1-CECD1-5-Artifact-9.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.11'], check=True)
import timm; print('timm', timm.__version__)


In [ ]:
# 4. 클래스 목록 — fastapi/main.py TASKS['facial'] 와 순서까지 같아야 한다.
# 이 리터럴은 tests/test_model_contract.py 가 노트북을 ast 로 읽어 대조한다. 실제 학습은 classes.py 를 쓰므로
# 여기서는 둘이 같은지 확인만 한다 — 어긋나면 배포 뒤 병명이 조용히 뒤바뀌는 사고가 나기 때문에 미리 막는다.
TASK = "facial"
CLASSES = ["acne", "atopic", "normal", "psoriasis", "rosacea", "seb_derm"]

import importlib.util
spec = importlib.util.spec_from_file_location('facial_classes', f'{REPO}/fastapi/training/aihub/facial/classes.py')
fc = importlib.util.module_from_spec(spec); spec.loader.exec_module(fc)
assert fc.TASK == TASK and list(fc.CLASSES) == CLASSES, f'노트북 {CLASSES} vs classes.py {list(fc.CLASSES)}'
print('클래스 6개 일치:', CLASSES)


In [ ]:
# 5. manifest — zip 을 /content 에 풀면서 만든다 (10GB, 수 분). 두 번째 실행부터는 이미 풀린 zip 을 건너뛴다.
# 끝에 'JSON diagnosis_name 교차검증 통과: 10,800장' 이 찍혀야 한다. ❌ 가 나오면 멈추고 예시 파일을 확인한다.
import subprocess, sys
r = subprocess.run([sys.executable, f'{REPO}/fastapi/training/aihub/facial/build_manifest.py', DATA_IN,
                    '--extract-to', EXTRACT, '--out', f'{RUN}/manifest.csv'], text=True)
assert r.returncode == 0, 'build_manifest 실패 — 위 출력의 ❌ 를 확인'


In [ ]:
# 6. 공식 홀드아웃 목록 — AI허브 Validation 1,200장을 그대로. 저장소 안의 공식 경로에 쓰고 Drive 에도 사본을 둔다.
# 이 파일은 커밋 대상이다(fastapi/tests/baselines/facial-holdout.csv). 이미 있으면 --force 없이는 덮어쓰지 않는다.
import subprocess, sys, shutil, os
HOLDOUT = f'{REPO}/fastapi/tests/baselines/facial-holdout.csv'
args = [sys.executable, f'{REPO}/fastapi/training/aihub/facial/make_holdout.py', '--manifest', f'{RUN}/manifest.csv', '--out', HOLDOUT]
r = subprocess.run(args, text=True)
assert r.returncode == 0 or os.path.exists(HOLDOUT), 'make_holdout 실패'
shutil.copy(HOLDOUT, f'{DRIVE}/facial-holdout.csv')
print('홀드아웃:', HOLDOUT, '→ Drive 사본 저장. 이 파일을 저장소에 커밋할 것')


In [ ]:
# 7. 학습 — fastapi/model.pth 백본을 물려받아(--init repo) 15 epoch. T4 에서 수십 분.
# 홀드아웃 목록의 사진은 절대 학습에 들어가지 않고 검증 세트로만 쓰인다. 리포트는 매 epoch 덮어쓰므로 세션이 끊겨도 곡선은 남는다.
import subprocess, sys
r = subprocess.run([sys.executable, f'{REPO}/fastapi/training/aihub/train.py',
                    '--manifest', f'{RUN}/manifest.csv', '--task', TASK, '--out-dir', RUN,
                    '--holdout-csv', HOLDOUT,
                    '--device', 'cuda', '--init', 'repo', '--epochs', '15', '--batch-size', '64',
                    '--image-size', '224', '--num-workers', '2'], text=True)
assert r.returncode == 0, 'train.py 실패'


In [ ]:
# 8. 산출물을 Drive 로 — 런타임이 끊기면 /content 는 사라진다.
import shutil, glob, os
for f in glob.glob(f'{RUN}/facial*'):
    shutil.copy(f, f'{DRIVE}/{os.path.basename(f)}')
print('Drive 저장:', sorted(os.path.basename(f) for f in glob.glob(f'{DRIVE}/facial*')))


## 다음 단계 (README §1-5)

1. Drive 의 `facial.pth` → 저장소 `fastapi/models/facial.pth`, `facial-train-report.json` → `fastapi/tests/baselines/facial-train-<날짜>.json`,
   `facial-holdout.csv` → `fastapi/tests/baselines/facial-holdout.csv` (6번 셀이 이미 그 경로에 썼다).
2. `cd fastapi && INTERNAL_API_SECRET=test python3 tests/test_model_contract.py` — `facial: 모델 출력 차원(6) == len(classes)(6)` 가 ✅ 로 바뀐다.
3. `python3 tests/evaluate.py --task facial --image-dir <풀린 폴더들> --csv tests/baselines/facial-holdout.csv --label-col label --json tests/baselines/facial-holdout-<날짜>.json`
   — 서빙 전처리로 홀드아웃 1,200장을 다시 재서 학습 리포트와 맞는지 본다.
4. `docker compose up --build` 로 띄워 진료 화면에서 분석 모델을 "안면부 피부질환"으로 바꿔 분석되는지 확인.
5. 커밋: `feat(fastapi): 안면부 피부질환 모델 가중치 및 기준 성적 추가`. 모델 버전은 가중치 파일 해시라 따로 손댈 것이 없다.

**결과를 읽을 때** — 합성 이미지로 학습·평가한 수치다. 실제 환자 사진에서의 성능은 보장하지 않으므로 화면에는 참고용임을 유지한다.
정면(얼굴 전체 1024²→224)과 측면(512²)의 recall 을 따로 본다(`facial-val-predictions.csv` 와 manifest 의 `view` 열을 조인).
